#Structured output 

In [1]:
import os 
from langchain.chat_models import init_chat_model
import dotenv
from dotenv import load_dotenv

load_dotenv()

os.environ["OPENROUTER_API_KEY"] = os.getenv("OPENROUTER_API_KEY")
os.environ["GEMINI_API_KEY"] = os.getenv("GEMINI_API_KEY")
os.environ['GROQ_API_KEY'] = os.getenv("GROQ_API_KEY")

model = init_chat_model(
    "gemini-2.5-flash",
    model_provider="google_genai"
)

response = model.invoke("What is the capital of France?")

print(response.content)

The capital of France is **Paris**.


In [13]:
#using pydantic to structure the output

import pydantic
from pydantic import BaseModel,Field

class CountryCapital(BaseModel):
    country: str = Field(description="Name of the country")
    capital: str = Field(description="Capital city of the country")

#nested structured output
class Person(BaseModel):
    name: str = Field(description="Name of the person")
    age: int = Field(description="Age of the person")
    favcountry: list[CountryCapital] = Field(description="Nationality of the person")
    gender: str = Field(description="Gender of the person")


response = model.invoke("What is the best country in the world? ")

print(response.content)

There isn't a single "best" country in the world, as "best" is highly subjective and depends entirely on what criteria someone values most. What's ideal for one person might not be for another.

However, many organizations attempt to rank countries based on various metrics. Here are some categories and countries that frequently rank highly:

*   **Quality of Life & Social Welfare (healthcare, education, safety, infrastructure):** Countries like **Norway, Denmark, Sweden, Finland, Switzerland, Canada, and the Netherlands** often top these lists. They typically have strong social safety nets, high levels of equality, and excellent public services.
*   **Happiness:** The **Nordic countries (Finland, Denmark, Iceland, Sweden, Norway)** consistently rank as the happiest nations in the World Happiness Report.
*   **Economic Opportunity & Stability:** **The United States, Germany, Japan, and Switzerland** are often cited for their strong economies, innovation, and business environments.
*   *

In [11]:
structured_response = model.with_structured_output(CountryCapital, include_raw ="True")

structured_response.invoke("What is the best country in the world? ")

{'raw': AIMessage(content='{\n  "country": "Cannot Determine",\n  "capital": "Cannot Determine"\n}', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019ea31b-991f-7c22-9e3e-5a44064638a3-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 11, 'output_tokens': 906, 'total_tokens': 917, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 885}}),
 'parsed': CountryCapital(country='Cannot Determine', capital='Cannot Determine'),
 'parsing_error': None}

In [14]:
person_response = model.with_structured_output(Person)
person_response.invoke("What is Modi's favorite country? ")


Person(name='Modi', age=73, favcountry=[], gender='Male')

In [17]:
from typing_extensions import Annotated, TypedDict

class CountryCapital(TypedDict):
    country: Annotated[str, "Name of the country"]
    capital: Annotated[str, "Capital city of the country"]

class Person(TypedDict):
    name: Annotated[str, "Name of the person"]
    age: Annotated[int, "Age of the person"]
    favcountry: Annotated[list[CountryCapital], "nationality of the person"]

person_response = model.with_structured_output(Person)
person_response.invoke("who is narendra modi? ")

{'name': 'Narendra Modi', 'age': 73, 'favcountry': []}